# YOLOV11 - small Training :

In [2]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")   # s = small


In [ ]:
model.train(
    data="data.yaml",
    epochs=50,
    imgsz=384,
    batch=16,
    workers=4,
    device=1,         
    patience=6,
    optimizer="AdamW",
    lr0=0.001,
    project="mealawe_project",
    name="yolo11_small",
    pretrained=True
)



In [ ]:
model.val()


# DETR-RESNET-50 training :

In [5]:
import torch
from torch.utils.data import DataLoader
from transformers import (
    DetrImageProcessor,
    DetrForObjectDetection,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset

C:\Users\PROFRESSOR\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
import json
import os
from PIL import Image
from torch.utils.data import Dataset

class COCODataset(Dataset):
    def __init__(self, img_dir, ann_path, processor):
        self.img_dir = img_dir
        self.processor = processor

        with open(ann_path) as f:
            coco = json.load(f)

        self.images = {img["id"]: img for img in coco["images"]}
        self.annotations = coco["annotations"]

        self.img_to_anns = {}
        for ann in self.annotations:
            self.img_to_anns.setdefault(ann["image_id"], []).append(ann)

        self.ids = list(self.images.keys())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_info = self.images[img_id]

        img_path = os.path.join(self.img_dir, img_info["file_name"])
        image = Image.open(img_path).convert("RGB")

        anns = self.img_to_anns.get(img_id, [])

        target = {
            "image_id": img_id,
            "annotations": anns
        }

        encoding = self.processor(
            images=image,
            annotations=target,
            return_tensors="pt"
        )

        return {k: v.squeeze(0) for k, v in encoding.items()}


In [13]:
from transformers import DetrImageProcessor

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")

train_ds = COCODataset(
    img_dir="coco_format_data/train/images",
    ann_path="coco_format_data/train/annotations.json",
    processor=processor
)

val_ds = COCODataset(
    img_dir="coco_format_data/valid/images",
    ann_path="coco_format_data/valid/annotations.json",
    processor=processor
)


In [ ]:
model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    num_labels=11,
    ignore_mismatched_sizes=True
)


In [ ]:
def collate_fn(batch):
    pixel_values = [item["pixel_values"] for item in batch]
    labels = [item["labels"] for item in batch]

    pixel_values = torch.stack(pixel_values)

    return {
        "pixel_values": pixel_values,
        "labels": labels
    }


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_ds,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    collate_fn=collate_fn
)


In [ ]:
# Training config
training_args = TrainingArguments(
    output_dir="./detr_mealbox",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=1e-4,
    weight_decay=1e-4,
    num_train_epochs=50,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,      # PyTorch Dataset
    eval_dataset=val_ds,         # PyTorch Dataset
    data_collator=collate_fn,    # VERY important
    tokenizer=processor
)

trainer.train()

In [ ]:
trainer.evaluate()


# Faster R-CNN training: